# What NASA Knew About If/Else That You Don't

Interactive companion to the [blog post](https://inferal.com/blog/what-nasa-knew-about-if-else).

Procedural logic buries intent. You can't inspect it, you can't explain it, and you can't change one rule without risking the rest. This notebook shows a different model: declarative rules that are readable, independently changeable, and self-cleaning.

**[clipspyx](https://github.com/inferal-oss/clipspyx)** is a Python DSL for [CLIPS](https://clipsrules.net), the inference engine NASA built in 1985.

In [ ]:
!pip install -q clipspyx[dsl,70x]

## Templates and rules

The clipspyx DSL parses Python class source at define-time. In notebooks, we write the definitions to a `.py` file and import them. This is the same code you'd write in a regular Python project.

**Templates** define the facts your system knows about. `Node`, `Pod`, and `Deployment` are cluster state. `Alert`, `Cordon`, `Evict`, and `Schedule` are derived facts that rules produce. Each template names exactly what it represents.

**Rules** are where intent lives. Each rule's docstring and patterns describe *when* and *why* it fires. Read `EvictPodsFromCordonedNode` below: a product manager or an on-call engineer can understand it without tracing control flow.

Notice `logical()` on the alert rules: this enables truth maintenance. If the supporting fact changes, the alert retracts itself automatically.

In [ ]:
%%writefile k8s_rules.py
from clipspyx import Environment
from clipspyx.dsl import Template, Rule

# --- Templates ---

class Node(Template):
    """A Kubernetes cluster node."""
    name: str
    status: str
    cpu_percent: float = 0.0
    memory_percent: float = 0.0

class Pod(Template):
    """A running pod in the cluster."""
    name: str
    namespace: str
    node: Node  # fact-address: which node this pod runs on
    status: str
    restarts: int = 0

class Deployment(Template):
    """A deployment managing a set of pods."""
    name: str
    namespace: str
    replicas: int
    available: int

class Alert(Template):
    """A derived alert, asserted by rules."""
    kind: str
    target: str
    message: str
    severity: str

class Cordon(Template):
    """Cordon a node to prevent new pod scheduling."""
    node_name: str
    reason: str

class Evict(Template):
    """Evict a pod from its current node."""
    pod_name: str
    reason: str

class Schedule(Template):
    """Schedule replacement pods for a deployment."""
    namespace: str
    deployment: str
    reason: str

# --- Rules ---

class NodeMemoryPressure(Rule):
    """Alert when a node's memory exceeds 90%."""
    logical(Node(name=name,
                memory_percent=mem and mem > 90))

    def __action__(self):
        Alert(__env__=self.__env__,
              kind="memory_pressure", target=self.name,
              message=f"Node {self.name} at {self.mem}% memory",
              severity="warning")

class NodeCriticalMemory(Rule):
    """Cordon a node when memory exceeds 95%."""
    n = Node(name=name,
             memory_percent=mem and mem > 95)

    def __action__(self):
        Cordon(__env__=self.__env__,
               node_name=self.name,
               reason=f"Memory critical: {self.mem}%")

class EvictPodsFromCordonedNode(Rule):
    """When a node is cordoned, evict its running pods."""
    Cordon(node_name=node_name)
    n = Node(name=node_name)
    p = Pod(node=n, status="running",
            name=pod_name, namespace=ns)

    def __action__(self):
        Evict(__env__=self.__env__,
              pod_name=self.pod_name,
              reason=f"Node {self.node_name} cordoned")

class RescheduleEvictedPod(Rule):
    """When a pod is evicted, schedule a replacement."""
    Evict(pod_name=pod_name)
    p = Pod(name=pod_name, namespace=ns)
    d = Deployment(namespace=ns, replicas=desired,
                   available=actual and actual < desired)

    def __action__(self):
        Schedule(__env__=self.__env__,
                 namespace=self.ns,
                 deployment=self.d.name,
                 reason=f"{self.actual}/{self.desired} available")

class CrashLoopDetection(Rule):
    """Alert on pods that keep restarting."""
    logical(Pod(name=name, namespace=ns,
                restarts=restarts and restarts > 5))

    def __action__(self):
        sev = "critical" if self.restarts > 10 else "warning"
        Alert(__env__=self.__env__,
              kind="crashloop", target=self.name,
              message=f"Pod {self.name}: {self.restarts} restarts",
              severity=sev)

class EvictCrashLoopPod(Rule):
    """Evict pods stuck in a severe crash loop."""
    p = Pod(name=name, namespace=ns,
            restarts=restarts and restarts > 10)

    def __action__(self):
        Evict(__env__=self.__env__,
              pod_name=self.name,
              reason=f"CrashLoopBackOff: {self.restarts} restarts")

In [ ]:
import importlib, k8s_rules
importlib.reload(k8s_rules)
from k8s_rules import *

## Run: assert facts and watch the cascade

We set up an environment, define all templates and rules, then assert cluster state. One call to `env.run()` triggers the full cascade:

1. node-1 at 97% memory → memory pressure alert + cordon action
2. cordon action → evict pods on node-1
3. eviction → reschedule pods for under-provisioned deployments
4. cache-c with 12 restarts → crashloop alert + eviction (independent chain)

In [ ]:
from IPython.display import HTML

env = Environment()
NodeAssert = env.define(Node)
PodAssert = env.define(Pod)
DeploymentAssert = env.define(Deployment)
env.define(Alert)
env.define(Cordon)
env.define(Evict)
env.define(Schedule)
env.define(NodeMemoryPressure)
env.define(NodeCriticalMemory)
env.define(EvictPodsFromCordonedNode)
env.define(RescheduleEvictedPod)
env.define(CrashLoopDetection)
env.define(EvictCrashLoopPod)

env.reset()

# Assert cluster state
node1 = NodeAssert(name="node-1", status="ready", cpu_percent=45.0, memory_percent=97.0)
node2 = NodeAssert(name="node-2", status="ready", cpu_percent=30.0, memory_percent=60.0)

PodAssert(name="api-server-a", namespace="production", node=node1, status="running")
PodAssert(name="worker-b", namespace="production", node=node1, status="running")
PodAssert(name="cache-c", namespace="staging", node=node2, status="running", restarts=12)

DeploymentAssert(name="api-server", namespace="production", replicas=3, available=2)
DeploymentAssert(name="worker", namespace="production", replicas=2, available=1)

env.run()

# --- Display helpers ---

SEVERITY_COLORS = {"critical": "#dc2626", "warning": "#d97706", "info": "#2563eb"}
FACT_COLORS = {"Cordon": "#7c3aed", "Evict": "#dc2626", "Schedule": "#059669"}

TARGET_SLOT = {"Cordon": "node_name", "Evict": "pod_name", "Schedule": "namespace"}

def badge(text, color):
    return (f'<span style="background:{color};color:white;padding:2px 8px;'
            f'border-radius:4px;font-size:12px;font-weight:600">{text}</span>')

def render_alerts(env):
    tpl = env.find_template(Alert.__clipspyx_dsl__.name)
    rows = ""
    for f in tpl.facts():
        color = SEVERITY_COLORS.get(f['severity'], '#6b7280')
        rows += (f"<tr><td>{badge(f['severity'], color)}</td>"
                 f"<td><strong>{f['kind']}</strong></td>"
                 f"<td><code>{f['target']}</code></td>"
                 f"<td>{f['message']}</td></tr>")
    if not rows:
        rows = '<tr><td colspan="4" style="text-align:center;color:#9ca3af">No alerts</td></tr>'
    return HTML(
        '<h3>\u26a0\ufe0f Alerts</h3>'
        '<table style="border-collapse:collapse;width:100%">'
        '<tr style="border-bottom:2px solid #e5e7eb">'
        '<th style="text-align:left;padding:6px">Severity</th>'
        '<th style="text-align:left;padding:6px">Kind</th>'
        '<th style="text-align:left;padding:6px">Target</th>'
        '<th style="text-align:left;padding:6px">Message</th></tr>'
        f'{rows}</table>')

def render_derived(env):
    rows = ""
    for cls in [Cordon, Evict, Schedule]:
        tpl = env.find_template(cls.__clipspyx_dsl__.name)
        color = FACT_COLORS.get(cls.__name__, '#6b7280')
        slot = TARGET_SLOT[cls.__name__]
        for f in tpl.facts():
            rows += (f"<tr><td>{badge(cls.__name__, color)}</td>"
                     f"<td><code>{f[slot]}</code></td>"
                     f"<td>{f['reason']}</td></tr>")
    if not rows:
        rows = '<tr><td colspan="3" style="text-align:center;color:#9ca3af">No derived facts</td></tr>'
    return HTML(
        '<h3>\u2699\ufe0f Derived Facts</h3>'
        '<table style="border-collapse:collapse;width:100%">'
        '<tr style="border-bottom:2px solid #e5e7eb">'
        '<th style="text-align:left;padding:6px">Type</th>'
        '<th style="text-align:left;padding:6px">Target</th>'
        '<th style="text-align:left;padding:6px">Reason</th></tr>'
        f'{rows}</table>')

display(render_alerts(env))
display(render_derived(env))

Because `NodeMemoryPressure` uses `logical()`, the alert is logically supported by the Node fact. Update the node's memory to a healthy level, and the alert cleans itself up. No reconciliation loop. No cleanup code.

In [ ]:
# Update node-1 memory from 97% to 60%
node1.modify_slots(memory_percent=60.0)
env.run()

# Check alerts — memory_pressure should be gone
alerts_tpl = env.find_template(Alert.__clipspyx_dsl__.name)
remaining = list(alerts_tpl.facts())

if not any(f['kind'] == 'memory_pressure' for f in remaining):
    display(HTML(
        '<div style="background:#f0fdf4;border:1px solid #86efac;border-radius:8px;padding:16px;margin:8px 0">'
        '<strong style="color:#166534">\u2713 Memory pressure alert was automatically retracted!</strong><br>'
        '<span style="color:#15803d">No cleanup code. No reconciliation loop. Truth maintenance handled it.</span>'
        '</div>'))

display(render_alerts(env))

## Try it yourself

Experiment with the rules above. Some ideas:

- **Add a node**: assert a third node with high CPU and write a `NodeHighCPU` rule
- **Change thresholds**: what happens if you lower the memory pressure threshold to 80%?
- **Add a new rule**: create a `DrainBeforeCordon` rule that asserts a drain Action before cordoning
- **Test truth maintenance**: retract cache-c and watch the crashloop alert disappear

The key insight: you never edit existing rules. You add new ones, and the engine figures out the interactions.

> **Note:** The DSL parses Python source files, so templates and rules must be written to a `.py` file (using `%%writefile`) and imported.

---

*Learn more at [inferal.com](https://inferal.com) — where we take these ideas to production scale.*